In [1]:
import sys
import json
from tqdm import tqdm
from openai import OpenAI 
import os
from functools import reduce
from typing import Dict
import gc
from gigachat import GigaChat 
from gigachat.models import Chat, Messages

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector
from src.utils.data_structs import TripletCreator, NodeCreator, Relation, NODES_TYPES_MAP, RELATIONS_TYPES_MAP
from src.llm_agent.agent_model import SYSTEM_PROMPT

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

DATASET_PATH = '../data/Augment_DiaASQ.json'
LOAD_EXTRACTED_TRIPLETS_FILE = ''
gc.collect()

0

In [ ]:
class GigaChatAgent:
    def __init__(self, creds: str, scope: str = 'GIGACHAT_API_CORP', model: str = "GigaChat-Pro",
                 verify_ssl_certs: bool = False) -> None:
        self.giga_model = GigaChat(
            credentials=creds, scope=scope, verify_ssl_certs=verify_ssl_certs, model=model) 
        self.system_prompt = SYSTEM_PROMPT

    def generate(self, user_prompt: str, assistant_prompt: str = None, 
                 system_prompt: str = None, gen_strategy: Dict = None) -> str:

        chat = Chat(messages=[Messages(role='system', content=system_prompt if system_prompt is not None else self.system_prompt), 
                            Messages(role='user', content=user_prompt)]) 
        response = self.giga_model.chat(chat) 
        return response.choices[0].message.content

In [ ]:
API_KEY = "OWUwOGUzOWEtMjJiNi00YmMxLThmMmItNzMwNjM2MTI2YmYxOjg2ODdiOTVhLTZkNDctNGFjOC1iMmViLTEyNDA5MmFiN2Q5Mw=="
agent = GigaChatAgent(creds=API_KEY)

In [ ]:
agent.generate("Сколько будет 2+2?")

### Extract

In [2]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

In [3]:
raw_texts = list(map(lambda v: v['text_dialog'], data['data']))
raw_time = list(map(lambda v: v['time'].split(',')[0], data['data']))
print(len(raw_texts), len(raw_time))

3483 3483


In [ ]:
#agent = AgentConnector.open()
#agent.generate("Сколько будет 2 + 2?")

In [ ]:
extractor = LLMExtractor(agent_conn=agent)

In [ ]:
extracted_triplets = []

In [ ]:
for i in tqdm(10):
    out = extractor.extract(raw_texts[i])
    extracted_triplets.append(out)
    print(raw_texts[i])
    print(out)
    print("===================")

In [ ]:
print(sum(list(map(len, extracted_triplets))))
with open("tmp_extracted_openai_gpt4omini_triplets.json", 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(extracted_triplets, ensure_ascii=False))

### Update

In [32]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://localhost:7687", user="neo4j", pwd="password", db_name="diaasq2"),
    embeddings_db=EmbeddingsDatabaseConnection(EmbeddingsDatabaseConnectionConfig(
        nodes_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_nodes/v10/densedb', 'vectorized_nodes', is_exist=False, need_to_clear=False
        ),
        triplets_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_triplets/v6/densedb', 'vectorized_triplets', is_exist=False, need_to_clear=False
        )
    ))
)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with MEAN pooling.


In [4]:
extracted_triplets = json.loads(open("../data/tmp_new_graph_extracted/tmp_extracted_openai_gpt4omini_triplets.json", 'r', encoding='utf-8').read())
print(len(extracted_triplets))

3483


In [13]:
# adding time
for group_idx in tqdm(range(len(extracted_triplets))):
    cur_time = raw_time[group_idx]
    for triplet_idx in range(len(extracted_triplets[group_idx])):
        if extracted_triplets[group_idx][triplet_idx][1]['prop']['type'] == 'simple':
            extracted_triplets[group_idx][triplet_idx][1]['prop']['time'] = cur_time
        else:
            extracted_triplets[group_idx][triplet_idx][2]['prop']['time'] = cur_time

100%|██████████| 3483/3483 [00:00<00:00, 43341.46it/s]


In [16]:
extracted_triplets = reduce(lambda acc, v: acc + v, extracted_triplets, [])
print(len(extracted_triplets))

283268


In [22]:
formated_triplets = []
for raw_triplet in tqdm(extracted_triplets):
    formated_triplets.append(TripletCreator.create(
        NodeCreator.create(name=raw_triplet[0]['name'], type=NODES_TYPES_MAP[raw_triplet[0]['type']], prop=raw_triplet[0]['prop'], add_stringified_node=False),
        Relation(name=raw_triplet[1]['name'], type=RELATIONS_TYPES_MAP[raw_triplet[1]['prop']['type']], prop={k: v for k, v in raw_triplet[1]['prop'].items() if k != 'type'}),
        NodeCreator.create(name=raw_triplet[2]['name'], type=NODES_TYPES_MAP[raw_triplet[2]['type']], prop=raw_triplet[2]['prop'], add_stringified_node=False)
    ))

100%|██████████| 283268/283268 [00:02<00:00, 138438.51it/s]


In [34]:
kg_model.graph_db.create_triplets(formated_triplets)

  0%|          | 0/283268 [00:00<?, ?it/s]Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: object)} {position: line: 1, column: 13, offset: 12} for query: 'MATCH (subj:object) WHERE subj.name = "gregory" RETURN elementID(subj) as id'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your

In [ ]:
kg_model.graph_db.close()